# Notebook 11: Ablation Study

**Project:** AareML — Predicting River Water Quality in Swiss Catchments  
**Dataset:** CAMELS-CH-Chem (Nascimento et al., 2025)  
**Focus gauge:** 2473 (Aare at Bern)  

---

Ablation study for the best Seq2Seq LSTM (hidden=256, n_layers=1, dropout=0.115, lr=3.16e-4, batch=64).
Each ablation trains a 3-seed ensemble (seeds 0, 42, 123) for 250 epochs with patience=25.

| Ablation | Conditions |
|----------|------------|
| A1 | Architecture: LSTM vs GRU |
| A2 | Teacher Forcing: tf=0.0 / tf=0.5 / tf=1.0 |
| A3 | Loss Function: MSE only vs NSE+MSE |
| A4 | Lookback Window: 7 / 14 / 21 days |

## 0 · Colab / UBELIX Setup

In [ ]:
# ── Colab setup (auto-runs only in Google Colab) ──────────────────────────
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    import os
    from pathlib import Path

    # ── 1. Clone repo ──────────────────────────────────────────────────────
    if not Path('AareML').exists():
        os.system('git clone https://github.com/polar-bear-after-lunch/AareML.git')
    if not str(Path.cwd()).endswith('AareML'):
        os.chdir('AareML')

    # ── 2. Install dependencies ────────────────────────────────────────────
    os.system('pip install -q -r requirements.txt')

    # ── 3. Mount Google Drive for persistent data storage ─────────────────
    from google.colab import drive
    drive.mount('/content/drive')

    DRIVE_DATA = Path('/content/drive/MyDrive/AareML_data')
    LOCAL_DATA = Path('data')
    LOCAL_DATA.mkdir(exist_ok=True)

    # ── 4. CAMELS-CH-Chem ─────────────────────────────────────────────────
    DRIVE_CAMELS = DRIVE_DATA / 'camels-ch-chem'
    LOCAL_CAMELS = LOCAL_DATA / 'camels-ch-chem'

    if DRIVE_CAMELS.exists():
        if not LOCAL_CAMELS.exists():
            os.system(f'ln -s {DRIVE_CAMELS} {LOCAL_CAMELS}')
        print('CAMELS-CH-Chem loaded from Google Drive.')
    else:
        print('Downloading CAMELS-CH-Chem to Google Drive (~165 MB, one-time)...')
        DRIVE_DATA.mkdir(parents=True, exist_ok=True)
        os.system(
            'wget -q --show-progress -O /tmp/camels.zip '
            '"https://zenodo.org/api/records/14980027/files/camels-ch-chem.zip/content"'
        )
        os.system(f'unzip -q /tmp/camels.zip -d {DRIVE_CAMELS}')
        os.system('rm /tmp/camels.zip')
        os.system(f'ln -s {DRIVE_CAMELS} {LOCAL_CAMELS}')
        print('CAMELS-CH-Chem saved to Google Drive for future sessions.')

    print(f'Setup complete. Working directory: {os.getcwd()}')

## 1 · Imports

In [ ]:
# ── CPU thread optimisation ────────────────────────────────────────────────
import os, multiprocessing
N_CPU = multiprocessing.cpu_count()
N_THREADS = min(N_CPU, 6)  # clamp to 6 — empirically optimal
os.environ['OMP_NUM_THREADS']      = str(N_THREADS)
os.environ['MKL_NUM_THREADS']      = str(N_THREADS)
os.environ['OPENBLAS_NUM_THREADS'] = str(N_THREADS)
print(f'CPU cores: {N_CPU} logical | Using {N_THREADS} threads for PyTorch')

In [ ]:
import sys, random, time
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from typing import Dict, List, Optional, Tuple

from sklearn.preprocessing import StandardScaler

from src.config  import *
from src.data    import load_gauge, preprocess, train_val_test_split, make_windows
from src.metrics import mean_rmse, nse, kge, metrics_table
from src.model   import (
    RiverDataset, Seq2SeqLSTM,
    train_model, predict, get_y_true,
)

sns.set_style('whitegrid')
plt.rcParams.update({'figure.dpi': 130})

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}  |  PyTorch: {torch.__version__}')

In [ ]:
# ── GPU / DataLoader configuration ────────────────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
NUM_WORKERS = 4 if DEVICE.type == 'cuda' else 0
PIN_MEMORY  = DEVICE.type == 'cuda'
print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'DataLoader workers: {NUM_WORKERS}, pin_memory: {PIN_MEMORY}')

# LOCAL_TEST: set True for 5-epoch debugging
LOCAL_TEST = False  # set True for 5-epoch debugging
if LOCAL_TEST:
    print('LOCAL_TEST mode: reduced epochs for fast CPU verification')

## 2 · Data Loading

In [ ]:
raw  = load_gauge(FOCUS_GAUGE)
data = preprocess(raw)
train, val, test = train_val_test_split(data)

# Training means used for NaN imputation — computed once, reused for all splits
train_means = (
    pd.concat([train[FEATURES].mean(), train[TARGETS].mean()])
    .groupby(level=0).first()   # deduplicate temp_sensor / O2C_sensor
)

print(f'Train: {train.index.min().date()} → {train.index.max().date()}  ({len(train):,} days)')
print(f'Val  : {val.index.min().date()}   → {val.index.max().date()}  ({len(val):,} days)')
print(f'Test : {test.index.min().date()}  → {test.index.max().date()}  ({len(test):,} days)')

## 3 · Window Creation and Scalers

In [ ]:
# ── Scalers (fitted on training data only — no leakage) ───────────────────
def impute(df, means):
    return df.fillna(means)

feat_scaler = StandardScaler().fit(impute(train[FEATURES], train_means[FEATURES]))
tgt_scaler  = StandardScaler().fit(impute(train[TARGETS],  train_means[TARGETS]))

def scale_windows(X_raw, y_raw):
    """Apply scalers to raw window arrays."""
    N, L, F = X_raw.shape
    X_s = feat_scaler.transform(X_raw.reshape(-1, F)).reshape(N, L, F).astype(np.float32)
    Nt, H, T = y_raw.shape
    y_s = tgt_scaler.transform(y_raw.reshape(-1, T)).reshape(Nt, H, T).astype(np.float32)
    return X_s, y_s

print('Feature means  (train):', dict(zip(FEATURES, feat_scaler.mean_.round(3))))
print('Target  scales (train):', dict(zip(TARGETS,  tgt_scaler.scale_.round(3))))

In [ ]:
# ── Default windows (LOOKBACK=21) ─────────────────────────────────────────
X_train, y_train, d_train = make_windows(train, train_means)
X_val,   y_val,   d_val   = make_windows(val,   train_means)
X_test,  y_test,  d_test  = make_windows(test,  train_means)

Xs_train, ys_train = scale_windows(X_train, y_train)
Xs_val,   ys_val   = scale_windows(X_val,   y_val)
Xs_test,  ys_test  = scale_windows(X_test,  y_test)

ds_train = RiverDataset(Xs_train, ys_train)
ds_val   = RiverDataset(Xs_val,   ys_val)
ds_test  = RiverDataset(Xs_test,  ys_test)

# Ground-truth targets in original units (used throughout)
y_true_test = get_y_true(ds_test, tgt_scaler)

print(f'Train windows: {len(ds_train):,}  |  Val: {len(ds_val):,}  |  Test: {len(ds_test):,}')

## 4 · Best-Model Configuration and Helper

In [ ]:
# ── Best hyperparameters from Optuna (nb03) ───────────────────────────────
BEST_HIDDEN   = 256
BEST_LAYERS   = 1
BEST_DROPOUT  = 0.115
BEST_LR       = 3.16e-4
BEST_BATCH    = 64
BEST_TF_START = 0.5    # teacher forcing start (Optuna best)

ENSEMBLE_SEEDS = [0, 42, 123]
EPOCHS    = 5   if LOCAL_TEST else 250
PATIENCE  = 3   if LOCAL_TEST else 25

print(f'Best params: hidden={BEST_HIDDEN}, layers={BEST_LAYERS}, '
      f'dropout={BEST_DROPOUT}, lr={BEST_LR:.2e}, batch={BEST_BATCH}')
print(f'Ensemble seeds: {ENSEMBLE_SEEDS}')
print(f'Epochs: {EPOCHS}, Patience: {PATIENCE}')

In [ ]:
def run_ablation(
    name: str,
    model_class,
    model_kwargs: dict,
    dl_train: DataLoader,
    dl_val: DataLoader,
    ds_test_: "RiverDataset",
    y_true_: np.ndarray,
    lr: float = BEST_LR,
    teacher_forcing_start: float = BEST_TF_START,
    loss_fn: str = 'nse_mse',   # 'nse_mse' or 'mse'
    verbose: bool = False,
) -> dict:
    """
    Train a 3-seed ensemble and return a metrics dict.

    Parameters
    ----------
    name          : identifier string for this condition
    model_class   : Seq2SeqLSTM or Seq2SeqGRU
    model_kwargs  : passed to model_class constructor
    dl_train      : training DataLoader
    dl_val        : validation DataLoader
    ds_test_      : test RiverDataset
    y_true_       : ground-truth [N, H, T] in original units
    lr            : learning rate
    teacher_forcing_start : teacher forcing start ratio
    loss_fn       : 'nse_mse' (default) or 'mse'
    verbose       : print per-epoch progress

    Returns
    -------
    dict with keys: name, y_pred, RMSE_DO, RMSE_temp, NSE_DO, NSE_temp,
                    KGE_DO, KGE_temp, mean_rmse
    """
    # ── Monkey-patch train_model for MSE-only variant ──────────────────────
    # We replicate train_model logic inline so we can swap the loss function
    # without modifying the shared src/model.py.

    def _train_one_seed(model, seed):
        torch.manual_seed(seed)
        np.random.seed(seed)

        model = model.to(DEVICE)
        optimiser = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimiser, mode='min', factor=0.5, patience=5
        )

        def nse_mse_loss(pred, target, alpha=0.5):
            mse_val = torch.mean((pred - target) ** 2)
            var     = torch.var(target, unbiased=False).clamp(min=1e-8)
            nse_loss = mse_val / var
            return alpha * nse_loss + (1.0 - alpha) * mse_val

        mse_criterion = nn.MSELoss()

        criterion = nse_mse_loss if loss_fn == 'nse_mse' else mse_criterion

        best_val   = np.inf
        best_state = None
        wait       = 0
        history    = {'train': [], 'val': []}

        for epoch in range(1, EPOCHS + 1):
            tf = teacher_forcing_start * max(0.0, 1.0 - epoch / (EPOCHS * 0.5))
            # For tf=1.0 (full teacher forcing, no decay), override
            if teacher_forcing_start == 1.0:
                tf = 1.0

            model.train()
            train_loss = 0.0
            for xb, yb in dl_train:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                optimiser.zero_grad()
                pred = model(xb, teacher_forcing_ratio=tf, y_target=yb)
                loss = criterion(pred, yb) if loss_fn == 'mse' else nse_mse_loss(pred, yb)
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimiser.step()
                train_loss += loss.item() * len(xb)
            train_loss /= len(dl_train.dataset)

            model.eval()
            val_loss = 0.0
            with torch.no_grad():
                for xb, yb in dl_val:
                    xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                    val_loss += (criterion(model(xb), yb) if loss_fn == 'mse'
                                 else nse_mse_loss(model(xb), yb)).item() * len(xb)
            val_loss /= len(dl_val.dataset)

            scheduler.step(val_loss)
            history['train'].append(train_loss)
            history['val'].append(val_loss)

            if val_loss < best_val:
                best_val   = val_loss
                best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
                wait = 0
            else:
                wait += 1
                if wait >= PATIENCE:
                    if verbose:
                        print(f'    [{name}|seed={seed}] Early stop epoch {epoch} '
                              f'(best_val={best_val:.5f})')
                    break

            if verbose and epoch % 50 == 0:
                print(f'    [{name}|seed={seed}] Epoch {epoch:3d} | '
                      f'train={train_loss:.5f}  val={val_loss:.5f}  '
                      f'tf={tf:.2f}')

        if best_state is None:
            raise RuntimeError(f'Training diverged for {name} seed={seed}')
        model.load_state_dict(best_state)
        return model, history

    # ── 3-seed ensemble ────────────────────────────────────────────────────
    seed_preds = []
    for s in tqdm(ENSEMBLE_SEEDS, desc=f'{name}', leave=False):
        torch.manual_seed(s)
        np.random.seed(s)
        m = model_class(**model_kwargs)
        m, _ = _train_one_seed(m, s)
        y_pred_s = predict(m, ds_test_, tgt_scaler, device=DEVICE)
        seed_preds.append(y_pred_s)

    y_pred = np.mean(seed_preds, axis=0)   # ensemble average

    rmse_d = mean_rmse(y_true_, y_pred)
    nse_d  = nse(y_true_,  y_pred)
    kge_d  = kge(y_true_,  y_pred)

    result = {
        'name':      name,
        'y_pred':    y_pred,
        'RMSE_DO':   rmse_d['O2C_sensor'],
        'RMSE_temp': rmse_d['temp_sensor'],
        'NSE_DO':    nse_d['O2C_sensor'],
        'NSE_temp':  nse_d['temp_sensor'],
        'KGE_DO':    kge_d['O2C_sensor'],
        'KGE_temp':  kge_d['temp_sensor'],
        'mean_rmse': float(np.mean([rmse_d['O2C_sensor'], rmse_d['temp_sensor']])),
    }
    print(f"  {name:40s}  RMSE_DO={result['RMSE_DO']:.4f}  "
          f"RMSE_temp={result['RMSE_temp']:.4f}  "
          f"NSE_DO={result['NSE_DO']:.3f}  "
          f"NSE_temp={result['NSE_temp']:.3f}")
    return result

## 5 · A1: Architecture — GRU vs LSTM

Compare LSTM encoder-decoder (best params) against an equivalent GRU.  
GRU has fewer parameters (no cell state) but similar inductive bias.

| Model | Encoder | Decoder |
|-------|---------|----------|
| LSTM (baseline) | nn.LSTM | nn.LSTM |
| GRU | nn.GRU | nn.GRU |

In [ ]:
class Seq2SeqGRU(nn.Module):
    """
    Encoder-decoder GRU for multi-step time-series forecasting.

    Mirrors Seq2SeqLSTM exactly but uses nn.GRU throughout.
    Key difference: GRU hidden state is a single tensor (not a (h, c) tuple),
    so the encoder/decoder interface is handled accordingly.

    Parameters
    ----------
    n_feat   : number of input features
    n_tgt    : number of output targets
    hidden   : GRU hidden size
    n_layers : number of stacked GRU layers
    dropout  : dropout probability (applied between GRU layers and before fc)
    """

    def __init__(self, n_feat: int = N_FEAT, n_tgt: int = N_TGT,
                 hidden: int = 64, n_layers: int = 2, dropout: float = 0.2):
        super().__init__()
        self.n_tgt   = n_tgt
        self.horizon = HORIZON

        enc_drop = dropout if n_layers > 1 else 0.0
        # GRU encoder — accepts same inputs as LSTM but returns (out, h)
        # instead of (out, (h, c)) — no cell state
        self.encoder = nn.GRU(
            input_size=n_feat, hidden_size=hidden,
            num_layers=n_layers, dropout=enc_drop, batch_first=True,
        )
        # GRU decoder — input is the previous output token (n_tgt features)
        self.decoder = nn.GRU(
            input_size=n_tgt, hidden_size=hidden,
            num_layers=n_layers, dropout=enc_drop, batch_first=True,
        )
        self.drop = nn.Dropout(dropout)
        self.fc   = nn.Linear(hidden, n_tgt)

    def forward(
        self,
        x: torch.Tensor,
        teacher_forcing_ratio: float = 0.0,
        y_target: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        """
        Parameters
        ----------
        x                     : [batch, lookback, n_feat]
        teacher_forcing_ratio : probability of using ground truth as decoder input
        y_target              : [batch, horizon, n_tgt] — required if tf_ratio > 0

        Returns
        -------
        [batch, horizon, n_tgt]
        """
        # Encoder: GRU returns (output, h_n) — no cell state unlike LSTM
        _, h = self.encoder(x)              # h: [n_layers, batch, hidden]

        batch = x.size(0)
        dec_in = torch.zeros(batch, 1, self.n_tgt, device=x.device)

        outputs = []
        for t in range(self.horizon):
            # Decoder step: GRU returns (out, h) — pass hidden state h (not tuple)
            out, h = self.decoder(dec_in, h)     # out: [batch, 1, hidden]
            pred = self.fc(self.drop(out))        # [batch, 1, n_tgt]
            outputs.append(pred)

            use_tf = (
                teacher_forcing_ratio > 0.0
                and y_target is not None
                and torch.rand(1).item() < teacher_forcing_ratio
            )
            dec_in = y_target[:, t : t + 1, :] if use_tf else pred.detach()

        return torch.cat(outputs, dim=1)          # [batch, horizon, n_tgt]


# Sanity check: verify GRU output shape
_gru = Seq2SeqGRU(N_FEAT, N_TGT, hidden=64, n_layers=1, dropout=0.1)
_x   = torch.randn(8, LOOKBACK, N_FEAT)
_y   = _gru(_x)
assert _y.shape == (8, HORIZON, N_TGT), f'GRU output shape mismatch: {_y.shape}'
n_gru  = sum(p.numel() for p in _gru.parameters())
n_lstm = sum(p.numel() for p in Seq2SeqLSTM(N_FEAT, N_TGT, hidden=64, n_layers=1).parameters())
print(f'GRU  params (hidden=64, layers=1): {n_gru:,}')
print(f'LSTM params (hidden=64, layers=1): {n_lstm:,}')
print(f'GRU output shape: {_y.shape}  ✓')
del _gru, _x, _y

In [ ]:
# ── A1: DataLoaders ────────────────────────────────────────────────────────
dl_train_a1 = DataLoader(ds_train, batch_size=BEST_BATCH, shuffle=True, drop_last=True,
                          num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
dl_val_a1   = DataLoader(ds_val,   batch_size=BEST_BATCH, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

arch_kwargs = dict(n_feat=N_FEAT, n_tgt=N_TGT,
                   hidden=BEST_HIDDEN, n_layers=BEST_LAYERS, dropout=BEST_DROPOUT)

print('=== A1: Architecture Ablation ===')
t0 = time.time()

res_a1_lstm = run_ablation(
    'A1_LSTM',
    Seq2SeqLSTM, arch_kwargs,
    dl_train_a1, dl_val_a1, ds_test, y_true_test,
)
res_a1_gru = run_ablation(
    'A1_GRU',
    Seq2SeqGRU, arch_kwargs,
    dl_train_a1, dl_val_a1, ds_test, y_true_test,
)
print(f'A1 done in {time.time()-t0:.1f}s')

## 6 · A2: Teacher Forcing — None vs Partial vs Full

| Condition | tf_start | Schedule |
|-----------|----------|----------|
| tf=0.0 | 0.0 | Pure autoregressive (no ground truth fed to decoder) |
| tf=0.5 | 0.5 | Linear decay 0.5→0 over first half of training (Optuna best) |
| tf=1.0 | 1.0 | Full teacher forcing — always feed ground truth, no decay |

In [ ]:
dl_train_a2 = DataLoader(ds_train, batch_size=BEST_BATCH, shuffle=True, drop_last=True,
                          num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
dl_val_a2   = DataLoader(ds_val,   batch_size=BEST_BATCH, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

print('=== A2: Teacher Forcing Ablation ===')
t0 = time.time()

res_a2_tf00 = run_ablation(
    'A2_TF=0.0  (no teacher forcing)',
    Seq2SeqLSTM, arch_kwargs,
    dl_train_a2, dl_val_a2, ds_test, y_true_test,
    teacher_forcing_start=0.0,
)
res_a2_tf05 = run_ablation(
    'A2_TF=0.5  (Optuna best / baseline)',
    Seq2SeqLSTM, arch_kwargs,
    dl_train_a2, dl_val_a2, ds_test, y_true_test,
    teacher_forcing_start=0.5,
)
res_a2_tf10 = run_ablation(
    'A2_TF=1.0  (full teacher forcing)',
    Seq2SeqLSTM, arch_kwargs,
    dl_train_a2, dl_val_a2, ds_test, y_true_test,
    teacher_forcing_start=1.0,
)
print(f'A2 done in {time.time()-t0:.1f}s')

## 7 · A3: Loss Function — MSE only vs NSE+MSE combined

| Condition | Loss |
|-----------|------|
| MSE only | `nn.MSELoss()` — standard squared error |
| NSE+MSE | `alpha*NSE_loss + (1-alpha)*MSE`, alpha=0.5 (current default) |

In [ ]:
dl_train_a3 = DataLoader(ds_train, batch_size=BEST_BATCH, shuffle=True, drop_last=True,
                          num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
dl_val_a3   = DataLoader(ds_val,   batch_size=BEST_BATCH, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

print('=== A3: Loss Function Ablation ===')
t0 = time.time()

res_a3_mse = run_ablation(
    'A3_Loss=MSE only',
    Seq2SeqLSTM, arch_kwargs,
    dl_train_a3, dl_val_a3, ds_test, y_true_test,
    loss_fn='mse',
)
res_a3_nse_mse = run_ablation(
    'A3_Loss=NSE+MSE  (alpha=0.5, baseline)',
    Seq2SeqLSTM, arch_kwargs,
    dl_train_a3, dl_val_a3, ds_test, y_true_test,
    loss_fn='nse_mse',
)
print(f'A3 done in {time.time()-t0:.1f}s')

## 8 · A4: Lookback Window — 7 vs 14 vs 21 days

Shorter lookbacks reduce memory requirements but may miss longer seasonal patterns.
Windows must be rebuilt for each lookback size.

| Condition | Lookback | Input shape |
|-----------|----------|-------------|
| 7 days | 7 | [N, 7, 4] |
| 14 days | 14 | [N, 14, 4] |
| 21 days (baseline) | 21 | [N, 21, 4] |

In [ ]:
def make_datasets_for_lookback(lb: int):
    """Build train/val/test RiverDatasets with a given lookback window."""
    X_tr, y_tr, _ = make_windows(train, train_means, lookback=lb)
    X_va, y_va, _ = make_windows(val,   train_means, lookback=lb)
    X_te, y_te, _ = make_windows(test,  train_means, lookback=lb)

    Xs_tr, ys_tr = scale_windows(X_tr, y_tr)
    Xs_va, ys_va = scale_windows(X_va, y_va)
    Xs_te, ys_te = scale_windows(X_te, y_te)

    ds_tr = RiverDataset(Xs_tr, ys_tr)
    ds_va = RiverDataset(Xs_va, ys_va)
    ds_te = RiverDataset(Xs_te, ys_te)

    y_true_ = get_y_true(ds_te, tgt_scaler)
    print(f'  lookback={lb}: train={len(ds_tr):,}  val={len(ds_va):,}  test={len(ds_te):,}')
    return ds_tr, ds_va, ds_te, y_true_


print('=== A4: Lookback Window Ablation ===')
t0 = time.time()

a4_results = {}
for lb in [7, 14, 21]:
    print(f'\nBuilding windows for lookback={lb}...')
    ds_tr_lb, ds_va_lb, ds_te_lb, y_true_lb = make_datasets_for_lookback(lb)

    # For A4 the model architecture does NOT change — lookback is implicit
    # in the sequence length fed to the encoder, not a model hyperparameter.
    dl_tr_lb = DataLoader(ds_tr_lb, batch_size=BEST_BATCH, shuffle=True, drop_last=True,
                           num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
    dl_va_lb = DataLoader(ds_va_lb, batch_size=BEST_BATCH, shuffle=False,
                           num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

    label = 'baseline' if lb == 21 else ''
    cond_name = f'A4_Lookback={lb:2d}d  ({label})' if label else f'A4_Lookback={lb:2d}d'

    res = run_ablation(
        cond_name,
        Seq2SeqLSTM, arch_kwargs,
        dl_tr_lb, dl_va_lb, ds_te_lb, y_true_lb,
    )
    a4_results[lb] = res

res_a4_lb7  = a4_results[7]
res_a4_lb14 = a4_results[14]
res_a4_lb21 = a4_results[21]
print(f'\nA4 done in {time.time()-t0:.1f}s')

## 9 · Results Summary Table

In [ ]:
# ── Collect all results into a flat list ──────────────────────────────────
all_results = [
    # A1
    res_a1_lstm,
    res_a1_gru,
    # A2
    res_a2_tf00,
    res_a2_tf05,
    res_a2_tf10,
    # A3
    res_a3_mse,
    res_a3_nse_mse,
    # A4
    res_a4_lb7,
    res_a4_lb14,
    res_a4_lb21,
]

ablation_group = {
    'A1_LSTM':                           'A1: Architecture',
    'A1_GRU':                            'A1: Architecture',
    'A2_TF=0.0  (no teacher forcing)':   'A2: Teacher Forcing',
    'A2_TF=0.5  (Optuna best / baseline)': 'A2: Teacher Forcing',
    'A2_TF=1.0  (full teacher forcing)':  'A2: Teacher Forcing',
    'A3_Loss=MSE only':                  'A3: Loss Function',
    'A3_Loss=NSE+MSE  (alpha=0.5, baseline)': 'A3: Loss Function',
    'A4_Lookback= 7d':                   'A4: Lookback Window',
    'A4_Lookback=14d':                   'A4: Lookback Window',
    'A4_Lookback=21d  (baseline)':       'A4: Lookback Window',
}

rows = []
for r in all_results:
    group = ablation_group.get(r['name'], 'Other')
    rows.append({
        'Ablation':   group,
        'Condition':  r['name'],
        'RMSE_DO':    round(r['RMSE_DO'],    4),
        'RMSE_temp':  round(r['RMSE_temp'],  4),
        'NSE_DO':     round(r['NSE_DO'],     3),
        'NSE_temp':   round(r['NSE_temp'],   3),
        'KGE_DO':     round(r['KGE_DO'],     3),
        'KGE_temp':   round(r['KGE_temp'],   3),
        'Mean_RMSE':  round(r['mean_rmse'],  4),
    })

results_df = pd.DataFrame(rows)
print(results_df.to_string(index=False))

## 10 · Figure: Grouped Bar Chart

In [ ]:
# ── Grouped bar chart: Mean RMSE across all ablation conditions ───────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

colors_a1 = ['#8B4FA8', '#BBA8CC']           # LSTM / GRU
colors_a2 = ['#D95F02', '#E8A977', '#A63B00']  # tf=0, 0.5, 1.0
colors_a3 = ['#2166AC', '#74ADD1']            # MSE / NSE+MSE
colors_a4 = ['#1A9641', '#78C679', '#ADDD8E']  # 7d / 14d / 21d

ablation_configs = [
    ('A1: Architecture',    [res_a1_lstm, res_a1_gru],            colors_a1),
    ('A2: Teacher Forcing', [res_a2_tf00, res_a2_tf05, res_a2_tf10], colors_a2),
    ('A3: Loss Function',   [res_a3_mse, res_a3_nse_mse],         colors_a3),
    ('A4: Lookback',        [res_a4_lb7, res_a4_lb14, res_a4_lb21], colors_a4),
]

# Short labels for readability
short_labels = {
    'A1_LSTM':                             'LSTM',
    'A1_GRU':                              'GRU',
    'A2_TF=0.0  (no teacher forcing)':     'TF=0.0',
    'A2_TF=0.5  (Optuna best / baseline)': 'TF=0.5*',
    'A2_TF=1.0  (full teacher forcing)':   'TF=1.0',
    'A3_Loss=MSE only':                    'MSE',
    'A3_Loss=NSE+MSE  (alpha=0.5, baseline)': 'NSE+MSE*',
    'A4_Lookback= 7d':                     'LB=7d',
    'A4_Lookback=14d':                     'LB=14d',
    'A4_Lookback=21d  (baseline)':         'LB=21d*',
}

for ax_i, (target_key, target_label) in enumerate([
    ('RMSE_DO', 'DO (mg/L)'), ('RMSE_temp', 'Temp (°C)')
]):
    ax = axes[ax_i]
    x_ticks = []
    x_labels = []
    x_pos = 0
    group_centers = []
    group_names = []

    for group_name, group_results, group_colors in ablation_configs:
        group_start = x_pos
        for res, color in zip(group_results, group_colors):
            val = res[target_key]
            bar = ax.bar(x_pos, val, color=color, width=0.7,
                         edgecolor='white', linewidth=0.5)
            ax.text(x_pos, val + 0.003, f'{val:.3f}',
                    ha='center', va='bottom', fontsize=7, rotation=0)
            x_ticks.append(x_pos)
            x_labels.append(short_labels.get(res['name'], res['name']))
            x_pos += 1
        group_center = (group_start + x_pos - 1) / 2
        group_centers.append(group_center)
        group_names.append(group_name)
        x_pos += 0.8   # gap between groups

    ax.set_xticks(x_ticks)
    ax.set_xticklabels(x_labels, rotation=30, ha='right', fontsize=8)
    ax.set_ylabel(f'RMSE — {target_label}', fontsize=11)
    ax.set_title(f'Ablation Study: {target_label}', fontsize=12)

    # Add group labels below x-axis
    for gc, gn in zip(group_centers, group_names):
        ax.annotate(gn, xy=(gc, 0), xycoords=('data', 'axes fraction'),
                    xytext=(0, -42), textcoords='offset points',
                    ha='center', fontsize=8, color='#444444',
                    fontweight='bold')

    # Mark baseline conditions with a dashed line (best tf=0.5, NSE+MSE, lb=21)
    ax.axhline(res_a1_lstm[target_key], color='grey', linewidth=0.7,
               linestyle='--', alpha=0.5, label='LSTM baseline')
    ax.legend(fontsize=8)

fig.suptitle(f'AareML Ablation Study — Gauge {FOCUS_GAUGE}\n'
             f'(* = baseline condition)', fontsize=13, y=1.01)
plt.tight_layout()
fig.savefig(FIGURES_DIR / '11_ablation_rmse_bar.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: figures/11_ablation_rmse_bar.png')

## 11 · Save Results

In [ ]:
# ── Save ablation results CSV ─────────────────────────────────────────────
import os as _os

# Save to both project results dir and workspace for downstream access
csv_paths = [
    RESULTS_DIR / 'ablation_results.csv',
    '/home/user/workspace/AareML/results/ablation_results.csv',
]

for p in csv_paths:
    _os.makedirs(_os.path.dirname(str(p)), exist_ok=True)
    results_df.to_csv(str(p), index=False)
    print(f'Saved → {p}')

print()
print('Final ablation results:')
print(results_df[['Ablation', 'Condition', 'RMSE_DO', 'RMSE_temp',
                   'NSE_DO', 'NSE_temp', 'Mean_RMSE']].to_string(index=False))

## Key Findings

### A1: Architecture — LSTM vs GRU

- **[Fill in after running]** Compare `RMSE_DO` and `RMSE_temp` for LSTM vs GRU.
- Expected: LSTM and GRU perform similarly on short river time series; LSTM may edge GRU
  slightly on DO (more complex seasonal dynamics) while GRU may match or exceed on temperature.
- GRU has ~25% fewer parameters due to the absence of a cell state — faster to train.
- If GRU matches LSTM: simpler architecture is sufficient for this task.

### A2: Teacher Forcing

- **[Fill in after running]** Compare the three conditions.
- `tf=0.0` (pure autoregressive): errors accumulate at longer horizons (exposure bias);
  expected to perform worst.
- `tf=0.5` (scheduled, Optuna best): balances training stability and inference robustness;
  expected to perform best.
- `tf=1.0` (full teacher forcing, no decay): training is easy but the model never learns
  to recover from its own errors; expected to degrade at longer horizons.
- The gap between tf=0.5 and tf=0.0 quantifies how much teacher forcing helps this dataset.

### A3: Loss Function

- **[Fill in after running]** Compare MSE-only vs NSE+MSE.
- NSE normalises by variance, penalising models that produce flat/biased predictions;
  expected to improve NSE metric but may trade off some RMSE.
- If NSE+MSE significantly outperforms MSE on NSE metric: the combined loss drives better
  distributional fit, which matters for extreme-event capture.
- If RMSE is similar but NSE is higher with NSE+MSE: confirms the benefit of the combined loss.

### A4: Lookback Window

- **[Fill in after running]** Compare 7d, 14d, 21d.
- Hypothesis: longer lookback captures more seasonality; diminishing returns after ~14d.
- If 21d ≈ 14d >> 7d: 14d may be sufficient (reduces compute).
- If 21d >> 14d ≈ 7d: longer memory is critical, validate even longer windows.
- River DO has strong diurnal/weekly cycles; a 7-day lookback should capture at least one
  full weekly cycle.

### Overall Recommendation

The ablations identify which design choices most affect predictive skill.  
Conditions where changing the component causes the largest RMSE degradation are the most
sensitive and thus most important to get right.